In [ ]:
import pandas as pd
import numpy as np
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import ipywidgets as widgets
from IPython.display import display
from pathlib import Path

# ── Find repo root (works locally AND on Binder) ──────────────────────────
def find_repo_root():
    current = Path.cwd()
    for candidate in [current] + list(current.parents):
        if (candidate / 'CITATION.cff').exists():
            return candidate
    return current.parent.parent

REPO_ROOT = find_repo_root()
GAMS = REPO_ROOT / 'dice' / 'validation' / 'gams_reference'
PY   = REPO_ROOT / 'dice' / 'validation' / 'python_output'

print(f'Repo root : {REPO_ROOT}')
print(f'GAMS path : {GAMS}  (exists: {GAMS.exists()})')
print(f'PY path   : {PY}  (exists: {PY.exists()})')

In [ ]:
# ══════════════════════════════════════════════════════════════════════════════
# SECTION 1 — SCENARIO FILE MAPPING
# ══════════════════════════════════════════════════════════════════════════════
scenario_files = {
    '1% Discounting'      : {'excel': str(GAMS/'dice_scenario_R1.xlsx'),    'state': str(PY/'dice2023_state_scen1.csv'),  'params': str(PY/'dice2023_parameters_scen1.csv')},
    '2% Discounting'      : {'excel': str(GAMS/'dice_scenario_R2.xlsx'),    'state': str(PY/'dice2023_state_scen2.csv'),  'params': str(PY/'dice2023_parameters_scen2.csv')},
    '3% Discounting'      : {'excel': str(GAMS/'dice_scenario_R3.xlsx'),    'state': str(PY/'dice2023_state_scen3.csv'),  'params': str(PY/'dice2023_parameters_scen3.csv')},
    '4% Discounting'      : {'excel': str(GAMS/'dice_scenario_R4.xlsx'),    'state': str(PY/'dice2023_state_scen4.csv'),  'params': str(PY/'dice2023_parameters_scen4.csv')},
    '5% Discounting'      : {'excel': str(GAMS/'dice_scenario_R5.xlsx'),    'state': str(PY/'dice2023_state_scen5.csv'),  'params': str(PY/'dice2023_parameters_scen5.csv')},
    'Max Temp 1.5\u00b0C' : {'excel': str(GAMS/'dice_scenario_T1.5.xlsx'), 'state': str(PY/'dice2023_state_scen6.csv'),  'params': str(PY/'dice2023_parameters_scen6.csv')},
    'Max Temp 2\u00b0C'   : {'excel': str(GAMS/'dice_scenario_T2.xlsx'),   'state': str(PY/'dice2023_state_scen7.csv'),  'params': str(PY/'dice2023_parameters_scen7.csv')},
    'Paris'               : {'excel': str(GAMS/'dice_scenario_paris.xlsx'), 'state': str(PY/'dice2023_state_scen8.csv'),  'params': str(PY/'dice2023_parameters_scen8.csv')},
    'Optimal'             : {'excel': str(GAMS/'dice_scenario_opt.xlsx'),   'state': str(PY/'dice2023_state_scen9.csv'),  'params': str(PY/'dice2023_parameters_scen9.csv')},
    'Base'                : {'excel': str(GAMS/'dice_scenario_base.xlsx'),  'state': str(PY/'dice2023_state_scen10.csv'), 'params': str(PY/'dice2023_parameters_scen10.csv')},
}

SCENARIO_COLORS = {
    '1% Discounting'      : '#1f77b4',
    '2% Discounting'      : '#ff7f0e',
    '3% Discounting'      : '#2ca02c',
    '4% Discounting'      : '#d62728',
    '5% Discounting'      : '#9467bd',
    'Max Temp 1.5\u00b0C' : '#8c564b',
    'Max Temp 2\u00b0C'   : '#e377c2',
    'Paris'               : '#7f7f7f',
    'Optimal'             : '#bcbd22',
    'Base'                : '#17becf',
}

In [ ]:
# ══════════════════════════════════════════════════════════════════════════════
# SECTION 2 — VARIABLE MAPPING
# ══════════════════════════════════════════════════════════════════════════════
var_map = {
    'Industrial CO2 GtCO2/yr'                         : ('state',  'EIND'),
    'Total CO2 Emissions, GTCO2/year'                  : ('state',  'ECO2'),
    'Total CO2e Emissions, GTCO2-E/year'               : ('state',  'ECO2E'),
    'Land emissions, GtCO2/year'                       : ('params', 'eland'),
    'Base abateable non-CO2 emission, GTCO2-E/year'    : ('params', 'CO2E_GHGabateB'),
    'Atmospheric concentration C (ppm)'                : ('state',  'CO2PPM'),
    'Atmospheric concentrations GtC'                   : ('state',  'MAT'),
    'Permanent C box'                                  : ('state',  'RES0'),
    'Slow C box'                                       : ('state',  'RES1'),
    'Medium C box'                                     : ('state',  'RES2'),
    'Fast C box'                                       : ('state',  'RES3'),
    'Cumulative CO2 emissions, GtC'                    : ('state',  'CCATOT'),
    'cacc'                                             : ('state',  'CACC'),
    'Atmospheric fraction CO2 since 1765'              : ('state',  'ATFRAC1765'),
    'Atmospheric fraction CO2 since 2020'              : ('state',  'ATFRAC2020'),
    'Alpha'                                            : ('state',  'ALPHA'),
    'Atmospheric temperaturer (deg c above preind)'    : ('state',  'TATM'),
    'Temp Box 1'                                       : ('state',  'TBOX1'),
    'Temp Box 2'                                       : ('state',  'TBOX2'),
    'Total forcings w/m2'                              : ('state',  'FORC'),
    'CO2 forcings w/m2'                                : ('state',  'FORC_CO2'),
    'Actual other abatable GHG forcings w/m2'          : ('state',  'F_GHGABATE'),
    'Output, net net trill 2019$'                      : ('state',  'Y'),
    'Output, gross-gross, 2019$'                       : ('state',  'YGROSS'),
    'Y gross-net, 2019$'                               : ('state',  'YNET'),
    'Capital stock, 2019$'                             : ('state',  'K'),
    'Gross investment, 2019$'                          : ('state',  'I'),
    'Savings rate, fraction gross output'              : ('state',  'Sopt'),
    'Consumption per capita, 2019$'                    : ('state',  'CPC'),
    'Consumption'                                      : ('state',  'C'),
    'Climate damages, fraction of output'              : ('state',  'DAMFRAC'),
    'Damages, 2019$'                                   : ('state',  'DAMAGES'),
    'Abatement, 2019$'                                 : ('state',  'ABATECOST'),
    'Abatement/0utput'                                 : ('state',  'ABATERAT'),
    'Carbon price (2019 $ per t CO2)'                  : ('state',  'CPRICE'),
    'Emissions control rate'                           : ('state',  'MIUopt'),
    'Social cost of carbon $/tCO2'                     : ('state',  'SCC'),
    'Cost, backstop technology ($/tCO2)'               : ('params', 'PBACKTIME'),
    'Population'                                       : ('state',  'L'),
    'TFP'                                              : ('state',  'AL'),
    'Change TFP, %/year'                               : ('params', 'gA'),
    'Sigmatot,(CO2/output, no controls, all CO2)'      : ('params', 'sigmatot'),
    'Short Interest rate, %/yr'                        : ('state',  'RSHORT'),
    'IFR'                                              : ('state',  'IRFT'),
    'Long interest rate(%)'                            : ('state',  'RLONG'),
    'RR'                                               : ('state',  'RR'),
    'RR1'                                              : ('params', 'RR1'),
}
var_map_clean = {k.strip(): v for k, v in var_map.items()}

In [ ]:
# ══════════════════════════════════════════════════════════════════════════════
# SECTION 3 — VARIABLE GROUPS
# ══════════════════════════════════════════════════════════════════════════════
var_groups = {
    'Climate & Temperature' : [
        'Atmospheric temperaturer (deg c above preind)',
        'Temp Box 1', 'Temp Box 2',
        'Atmospheric concentrations GtC',
        'Atmospheric concentration C (ppm)',
        'Total forcings w/m2', 'CO2 forcings w/m2',
        'Actual other abatable GHG forcings w/m2',
        'Atmospheric fraction CO2 since 1765',
        'Atmospheric fraction CO2 since 2020',
    ],
    'Emissions' : [
        'Industrial CO2 GtCO2/yr',
        'Total CO2 Emissions, GTCO2/year',
        'Total CO2e Emissions, GTCO2-E/year',
        'Land emissions, GtCO2/year',
        'Base abateable non-CO2 emission, GTCO2-E/year',
        'Cumulative CO2 emissions, GtC', 'cacc',
        'Permanent C box', 'Slow C box', 'Medium C box', 'Fast C box',
        'Alpha',
    ],
    'Economics' : [
        'Output, net net trill 2019$', 'Output, gross-gross, 2019$',
        'Y gross-net, 2019$', 'Capital stock, 2019$',
        'Gross investment, 2019$', 'Savings rate, fraction gross output',
        'Consumption per capita, 2019$', 'Consumption',
        'Climate damages, fraction of output',
        'Damages, 2019$', 'Abatement, 2019$', 'Abatement/0utput',
    ],
    'Policy & Prices' : [
        'Carbon price (2019 $ per t CO2)',
        'Emissions control rate',
        'Social cost of carbon $/tCO2',
        'Cost, backstop technology ($/tCO2)',
    ],
    'Population & TFP' : [
        'Population', 'TFP', 'Change TFP, %/year',
        'Sigmatot,(CO2/output, no controls, all CO2)',
    ],
    'Finance & Discounting' : [
        'Short Interest rate, %/yr', 'IFR',
        'Long interest rate(%)', 'RR', 'RR1',
    ],
}

In [ ]:
# ══════════════════════════════════════════════════════════════════════════════
# SECTION 4 — LOAD ALL DATA
# ══════════════════════════════════════════════════════════════════════════════
scenarios = {}
for scen_name, files in scenario_files.items():
    try:
        excel = pd.read_excel(files['excel'])
        excel.columns = excel.columns.str.strip()
        excel = excel.set_index('Period')
        scenarios[scen_name] = {
            'excel'  : excel,
            'state'  : pd.read_csv(files['state']).set_index('IPERIOD'),
            'params' : pd.read_csv(files['params']).set_index('PERIOD'),
        }
        print(f'Loaded : {scen_name}')
    except FileNotFoundError as ex:
        print(f'WARNING \u2014 file not found : {ex}')

periods    = next(iter(scenarios.values()))['excel'].index.tolist()
scen_names = list(scenarios.keys())
print(f'\nTotal scenarios loaded : {len(scenarios)}')
print(f'Total variables        : {len(var_map_clean)}')

In [ ]:
# ══════════════════════════════════════════════════════════════════════════════
# SECTION 5 — WIDGETS
# ══════════════════════════════════════════════════════════════════════════════
w_group = widgets.Dropdown(
    options=list(var_groups.keys()), value=list(var_groups.keys())[0],
    description='Group:', style={'description_width': '60px'},
    layout=widgets.Layout(width='400px'),
)
w_var = widgets.Dropdown(
    options=var_groups[w_group.value], value=var_groups[w_group.value][0],
    description='Variable:', style={'description_width': '60px'},
    layout=widgets.Layout(width='700px'),
)
scen_checkboxes = {
    scen: widgets.Checkbox(value=True, description=scen,
                            style={'description_width': 'initial'},
                            layout=widgets.Layout(width='200px'))
    for scen in scen_names
}
checkbox_row1 = widgets.HBox([scen_checkboxes[s] for s in scen_names[:5]])
checkbox_row2 = widgets.HBox([scen_checkboxes[s] for s in scen_names[5:]])
btn_all  = widgets.Button(description='Select All',   button_style='info',    layout=widgets.Layout(width='120px'))
btn_none = widgets.Button(description='Deselect All', button_style='warning', layout=widgets.Layout(width='120px'))
btn_all.on_click(lambda b: [setattr(cb, 'value', True)  for cb in scen_checkboxes.values()])
btn_none.on_click(lambda b: [setattr(cb, 'value', False) for cb in scen_checkboxes.values()])
out = widgets.Output()

In [ ]:
# ══════════════════════════════════════════════════════════════════════════════
# SECTION 6 — PLOT FUNCTION
# ══════════════════════════════════════════════════════════════════════════════
def plot_comparison(var, active_scenarios):
    src, csv_col = var_map_clean[var]
    ref_vals = []
    for sn in active_scenarios:
        if sn in scenarios:
            try: ref_vals.extend(scenarios[sn]['excel'][var].dropna().tolist())
            except KeyError: pass
    use_pct     = np.median(np.abs(ref_vals)) >= 10 if ref_vals else True
    diff_label  = '% Difference vs Nordhaus' if use_pct else 'Absolute Difference vs Nordhaus'
    diff_suffix = '%' if use_pct else ''

    fig = make_subplots(
        rows=2, cols=1, shared_xaxes=True,
        subplot_titles=(f'{var} \u2014 Nordhaus (dotted) vs PyDICE-2023 (solid)', diff_label),
        row_heights=[0.65, 0.35], vertical_spacing=0.10,
    )
    for sn in active_scenarios:
        if sn not in scenarios: continue
        color = SCENARIO_COLORS[sn]
        try:
            e = scenarios[sn]['excel'][var].values
            c = scenarios[sn]['state'][csv_col].values if src == 'state' else scenarios[sn]['params'][csv_col].values
            with np.errstate(divide='ignore', invalid='ignore'):
                diff = np.where(np.abs(e) > 1e-10, (c-e)/np.abs(e)*100, 0.0) if use_pct else (c-e)
        except KeyError:
            print(f"WARNING \u2014 '{var}' not found for {sn}"); continue

        fig.add_trace(go.Scatter(x=periods, y=e, name=f'{sn} \u2014 Nordhaus',
            line=dict(color=color, width=2, dash='dot'), legendgroup=sn,
            hovertemplate=f'<b>{sn} Nordhaus</b><br>Period: %{{x}}<br>Value: %{{y:.4f}}<extra></extra>'), row=1, col=1)
        fig.add_trace(go.Scatter(x=periods, y=c, name=f'{sn} \u2014 PyDICE-2023',
            line=dict(color=color, width=2), legendgroup=sn,
            hovertemplate=f'<b>{sn} PyDICE-2023</b><br>Period: %{{x}}<br>Value: %{{y:.4f}}<extra></extra>'), row=1, col=1)
        fig.add_trace(go.Scatter(x=periods, y=diff, name=sn,
            line=dict(color=color, width=1.5), fill='tozeroy',
            legendgroup=sn, showlegend=False,
            hovertemplate=f'<b>{sn}</b><br>Period: %{{x}}<br>Diff: %{{y:.4f}}{diff_suffix}<extra></extra>'), row=2, col=1)

    fig.add_trace(go.Scatter(x=periods, y=[0]*len(periods),
        line=dict(color='black', width=1, dash='dot'), showlegend=False), row=2, col=1)
    fig.update_layout(
        height=700, hovermode='x unified', plot_bgcolor='white', paper_bgcolor='white',
        margin=dict(r=200),
        legend=dict(x=1.02, y=1, tracegroupgap=5, bordercolor='#cccccc', borderwidth=1),
        xaxis2=dict(title='Period'), yaxis=dict(title=var),
        yaxis2=dict(title=diff_label, ticksuffix=diff_suffix,
                    zeroline=True, zerolinecolor='black', zerolinewidth=1),
    )
    fig.update_xaxes(showgrid=True, gridcolor='#eeeeee')
    fig.update_yaxes(showgrid=True, gridcolor='#eeeeee')
    fig.show()

In [ ]:
# ══════════════════════════════════════════════════════════════════════════════
# SECTION 7 — LINK WIDGETS & DISPLAY
# ══════════════════════════════════════════════════════════════════════════════
def refresh_plot(*args):
    active = [s for s, cb in scen_checkboxes.items() if cb.value]
    out.clear_output(wait=True)
    with out: plot_comparison(w_var.value, active)

def update_var_options(change):
    w_var.options = var_groups[change['new']]
    w_var.value   = var_groups[change['new']][0]

w_group.observe(update_var_options, names='value')
w_group.observe(refresh_plot, names='value')
w_var.observe(refresh_plot,   names='value')
for cb in scen_checkboxes.values(): cb.observe(refresh_plot, names='value')

display(widgets.VBox([
    widgets.HTML('<h3>DICE-2023 \u2014 Nordhaus (GAMS) vs PyDICE-2023 Validation Dashboard</h3>'),
    widgets.HBox([w_group, w_var]),
    widgets.HTML('<b>Scenarios \u2014 check/uncheck to toggle:</b>'),
    checkbox_row1, checkbox_row2,
    widgets.HBox([btn_all, btn_none]),
    out,
]))

refresh_plot()